# SBi-FAGAT v3 — Cloud GPU Benchmark Runner (Google Colab T4/L4/A100 / Kaggle GPU)

Offloads `dual_ce_{vn,pheme}` Cross-Encoder distillation recovery, full 5-seed Tier 1 sweeps (15+ models including `Calibrated Early-K Persistence`, `Ridge`, `LightGBM`, `FriedkinJohnsen`, `CasCN`, `CasFlow`, `BiEdgeGATv2`, `SBiFAGAT`, `SBiFAGAT + Tier2Prior` across `canonical`, `embedding_cluster`, `temporal`, and `loo` splits), Ablation & $K$-Curve, E2E raw-text efficiency (`--include-e2e`), and Tier 2 macro + 10-model signed link benchmarks to Colab/Kaggle GPU.

All outputs are written strictly into **`results/`** and **`checkpoints/`** (never overwriting `paper/`).

### Quick Start
1. **Colab:** Select `Runtime -> Change runtime type -> T4 GPU` (or L4/A100). Upload `data/processed/{vn,pheme}/` (`nodes.parquet`, `edges.parquet`, `nlp_features.pt`, `topic_splits.json`) to Google Drive (`MyDrive/SBiFAGAT/data/processed/{vn,pheme}`), or upload `/content/sbifagat_{vn,pheme}.zip`.
2. **Kaggle:** Enable `Accelerator -> GPU T4` and attach your processed dataset under `/kaggle/input/...`.
3. Run all cells top-to-bottom. At the end, download `sbifagat_v3_gpu_results.zip` and extract into your local repo root (`results/` and `checkpoints/`).

In [ ]:
import os
import shutil
import subprocess
import sys
from pathlib import Path

REPO_URL = "https://github.com/hieu-kiien/gnn-nlp-propagation.git"
REPO_BRANCH = os.environ.get("SBIF_BRANCH", "main")

is_kaggle = Path("/kaggle/working").exists()
work_root = Path("/kaggle/working/gnn-nlp-propagation") if is_kaggle else Path("/content/gnn-nlp-propagation")

if not Path("scripts/run_experiments.py").exists():
    if work_root.exists():
        os.chdir(work_root)
        subprocess.run(["git", "fetch", "--all"], check=True)
        subprocess.run(["git", "checkout", "-B", REPO_BRANCH, f"origin/{REPO_BRANCH}"], check=True)
    else:
        subprocess.run(["git", "clone", "-b", REPO_BRANCH, REPO_URL, str(work_root)], check=True)
        os.chdir(work_root)

if shutil.which("nvidia-smi"):
    subprocess.run(["nvidia-smi"], check=False)
for mod_name, pip_name in (
    ("torch_geometric", "torch-geometric"),
    ("transformers", "transformers"),
    ("datasets", "datasets"),
    ("sentencepiece", "sentencepiece"),
    ("google.protobuf", "protobuf"),
    ("huggingface_hub", "huggingface-hub"),
    ("umap", "umap-learn"),
    ("pydantic", "pydantic"),
    ("pyarrow", "pyarrow"),
    ("scipy", "scipy"),
    ("sklearn", "scikit-learn"),
    ("matplotlib", "matplotlib"),
    ("seaborn", "seaborn"),
    ("tqdm", "tqdm"),
    ("pandas", "pandas"),
    ("lightgbm", "lightgbm"),
    ("statsmodels", "statsmodels"),
):
    try:
        __import__(mod_name)
    except ImportError:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", pip_name], check=True)

import torch
print("Working dir:", Path.cwd())
print("PyTorch:", torch.__version__, "CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU device:", torch.cuda.get_device_name(0))


In [ ]:
import shutil
import zipfile
from pathlib import Path

# Adjust GDRIVE_ROOT if your staged SBiFAGAT folder lives elsewhere in Google Drive
GDRIVE_ROOT = Path("/content/drive/MyDrive/SBiFAGAT")
GDRIVE_DATA_ROOT = GDRIVE_ROOT / "data" / "processed"
REQUIRED_FILES = ("nodes.parquet", "edges.parquet", "nlp_features.pt")
OPTIONAL_FILES = ("topic_splits.json",)
ALL_CORPUS_FILES = set(REQUIRED_FILES) | set(OPTIONAL_FILES)

for _corpus, _zip_p in (("vn", Path("/content/sbifagat_vn.zip")), ("pheme", Path("/content/sbifagat_pheme.zip"))):
    _parts = sorted(Path("/content").glob(f"{_zip_p.name}.part*"))
    if _parts and not _zip_p.exists():
        with open(_zip_p, "wb") as _out:
            for _part in _parts:
                _out.write(_part.read_bytes())
        for _part in _parts:
            _part.unlink(missing_ok=True)
        print(f"[{_corpus}] Reassembled {len(_parts)} parts into {_zip_p}")
    if _zip_p.is_file():
        _dst = Path("data/processed") / _corpus
        _dst.mkdir(parents=True, exist_ok=True)
        if not all((_dst / f).exists() for f in REQUIRED_FILES):
            with zipfile.ZipFile(_zip_p, "r") as _zf:
                for _member in _zf.infolist():
                    if _member.is_dir():
                        continue
                    _base = Path(_member.filename).name
                    if _base in ALL_CORPUS_FILES:
                        with _zf.open(_member, "r") as _src, open(_dst / _base, "wb") as _out:
                            shutil.copyfileobj(_src, _out)
            print(f"[{_corpus}] Extracted uploaded bundle from {_zip_p} into {_dst}")

IS_KAGGLE = Path("/kaggle/working").exists()
IN_COLAB = False
if not IS_KAGGLE:
    try:
        from google.colab import drive  # type: ignore
        IN_COLAB = True
        if (
            not Path("data/processed/vn/nlp_features.pt").exists()
            and not Path("/content/sbifagat_vn.zip").exists()
            and not Path("/content/drive/MyDrive").exists()
        ):
            drive.mount("/content/drive")
    except (ImportError, NotImplementedError, RuntimeError):
        IN_COLAB = False


def _stage_corpus(corpus: str) -> bool:
    dst_dir = Path("data/processed") / corpus
    dst_dir.mkdir(parents=True, exist_ok=True)
    if all((dst_dir / f).exists() for f in REQUIRED_FILES):
        print(f"[{corpus}] Already staged at {dst_dir}")
        return True

    search_roots = [GDRIVE_DATA_ROOT / corpus, Path("/content"), Path("/kaggle/input")]
    found: dict[str, Path] = {}
    nlp_candidates: list[Path] = []
    for root in search_roots:
        if not root.exists():
            continue
        for p in sorted(root.rglob("nlp_features.pt")):
            if not p.is_file():
                continue
            parts_low = [part.lower() for part in p.parts]
            if root == (GDRIVE_DATA_ROOT / corpus) or corpus in parts_low or any(corpus in part for part in parts_low[-3:-1]):
                nlp_candidates.append(p)
    for nlp_p in nlp_candidates:
        src_dir = nlp_p.parent
        found = {"nlp_features.pt": nlp_p}
        for fname in ALL_CORPUS_FILES:
            cand = src_dir / fname
            if cand.is_file():
                found[fname] = cand
        if all(f in found for f in REQUIRED_FILES):
            break
    if not all(f in found for f in REQUIRED_FILES) and nlp_candidates:
        for root in search_roots:
            if not root.exists():
                continue
            for p in sorted(root.rglob("*")):
                if not p.is_file() or p.name not in ALL_CORPUS_FILES:
                    continue
                parts_low = [part.lower() for part in p.parts]
                if root == (GDRIVE_DATA_ROOT / corpus) or corpus in parts_low or any(corpus in part for part in parts_low[-3:-1]):
                    found.setdefault(p.name, p)
    if all(f in found for f in REQUIRED_FILES):
        for fname, src_p in found.items():
            tgt_p = dst_dir / fname
            if not tgt_p.exists():
                try:
                    tgt_p.symlink_to(src_p)
                except OSError:
                    shutil.copy2(src_p, tgt_p)
        print(f"[{corpus}] Linked/staged from {found['nlp_features.pt'].parent}")
        return True

    print(f"[{corpus}] Not found locally; copy {REQUIRED_FILES} into {dst_dir.resolve()} if running this track.")
    return False


# Optional: stage interim parquets & checkpoints/visobert_vihsd from Google Drive if RUN_TEACHER_RESCORE=True
for _rel_opt in ("data/interim/vn", "data/interim/pheme", "checkpoints/visobert_vihsd"):
    _src_opt = GDRIVE_ROOT / _rel_opt
    _dst_opt = Path(_rel_opt)
    if _src_opt.is_dir() and not _dst_opt.exists():
        _dst_opt.parent.mkdir(parents=True, exist_ok=True)
        try:
            _dst_opt.symlink_to(_src_opt)
        except OSError:
            shutil.copytree(_src_opt, _dst_opt, dirs_exist_ok=True)
        print(f"[Optional Stage] Linked/copied {_rel_opt} from Google Drive")

vn_ready = _stage_corpus("vn")
pheme_ready = _stage_corpus("pheme")
print("Dataset availability -> VN:", vn_ready, "| PHEME:", pheme_ready)


In [ ]:
import gc
import json
import subprocess
import sys
from pathlib import Path
import torch

# Distill missing dual_ce_vn / dual_ce_pheme checkpoints on GPU from staged data/processed/{vn,pheme}
# (zero test leakage via topic_splits.json; auto-skips if dual_head_weights.pt is already complete)
RUN_NLP_DISTILL_RECOVERY = True
# Optional: Set True only if you staged data/interim/{vn,pheme} and want to re-run Teacher NLI + Router + SpanPool
RUN_TEACHER_RESCORE = False

from src.nlp.dual_edge_scorer import (
    TWITTER_ROBERTA_MODEL_NAME,
    VISOBERT_MODEL_NAME,
    _has_complete_dual_ce_checkpoint,
    _has_hf_weight_files,
    distill_dual_cross_encoder,
    extract_parent_reply_pairs_from_parquet,
    split_pairs_by_inductive_topics,
)

if RUN_TEACHER_RESCORE:
    for _ds, _lang, _backbone, _min_rep, _k in (
        ("vn", "vi", VISOBERT_MODEL_NAME, 25, 20),
        ("pheme", "en", TWITTER_ROBERTA_MODEL_NAME, 15, 10),
    ):
        _interim = Path("data/interim") / _ds
        _proc = Path("data/processed") / _ds
        _ckpt = Path("checkpoints") / f"dual_ce_{_ds}"
        if not (_interim / "nodes.parquet").is_file():
            print(f"[SKIP Teacher:{_ds}] Missing {_interim}/nodes.parquet")
            continue
        _scorer_cmd = [
            sys.executable, "-u", "-m", "src.nlp.dual_edge_scorer",
            "--lang", _lang,
            "--interim-dir", str(_interim),
            "--vihsd-dir", "checkpoints/visobert_vihsd",
            "--output-dir", str(_ckpt),
            "--scored-cache-json", str(_interim / "scored_pairs.json"),
            "--teacher-backend", "dual_nli",
            "--hostility-target-router",
            "--min-total-replies", str(_min_rep),
            "--edge-dim", "2",
            "--student-backbone", _backbone,
            "--include-root-context",
            "--score-only", "--distill",
        ]
        if _ds == "vn" and not _has_hf_weight_files(Path("checkpoints/visobert_vihsd")):
            _scorer_cmd.append("--train-vihsd")
        subprocess.run(_scorer_cmd, check=True)
        subprocess.run([
            sys.executable, "-u", "-m", "src.nlp.extract_features",
            "--dataset", _ds,
            "--interim-dir", str(_interim),
            "--output-dir", str(_proc),
            "--scored-cache-json", str(_interim / "scored_pairs.json"),
            "--cross-encoder-ckpt", str(_ckpt),
            "--vihsd-dir", "checkpoints/visobert_vihsd",
            "--edge-dim", "2",
            "--node-backbone", _backbone,
            "--k-horizon", str(_k),
            "--min-total-replies", str(_min_rep),
            "--include-root-context",
        ], check=True)

if RUN_NLP_DISTILL_RECOVERY:
    for _ds, _lang, _backbone, _min_rep in (
        ("vn", "vi", VISOBERT_MODEL_NAME, 25),
        ("pheme", "en", TWITTER_ROBERTA_MODEL_NAME, 15),
    ):
        _proc = Path("data/processed") / _ds
        _ckpt = Path("checkpoints") / f"dual_ce_{_ds}"
        if not (_proc / "nlp_features.pt").is_file() or not (_proc / "topic_splits.json").is_file():
            print(f"[SKIP CE:{_ds}] Missing {_proc}/nlp_features.pt or topic_splits.json")
            continue
        if _has_complete_dual_ce_checkpoint(_ckpt, require_distill_report=True):
            print(f"[SKIP CE:{_ds}] Complete checkpoint (incl. dual_head_weights.pt) already exists in {_ckpt}")
            continue
        _pairs = extract_parent_reply_pairs_from_parquet(_proc, min_total_replies=_min_rep)
        _pt = torch.load(_proc / "nlp_features.pt", map_location="cpu", weights_only=False)
        _ts, _th = _pt["teacher_edge_stance"], _pt["teacher_edge_hostility"]
        _tr = _pt.get("teacher_node_root_stance") or {}
        for _p in _pairs:
            _eid = str(_p["edge_id"])
            _p["s_uv"] = float(_ts[_eid])
            _p["h_uv"] = float(_th[_eid])
            _p["root_s_uv"] = float(_tr.get(str(_p["target_id"]), _p["s_uv"]))
        del _pt
        gc.collect()
        _splits = split_pairs_by_inductive_topics(
            _pairs,
            canonical_topic_splits=json.loads((_proc / "topic_splits.json").read_text(encoding="utf-8")),
        )
        distill_dual_cross_encoder(
            _splits["train"],
            _splits["val"],
            output_dir=_ckpt,
            lang=_lang,
            model_name=_backbone,
            edge_dim=2,
            include_root_context=True,
            epochs=3,
            batch_size=16,
            lr=2e-5,
            max_length=256,
            distill_temperature=2.0,
            augment_root_pairs=True,
            seed=42,
        )


In [ ]:
import subprocess
import sys
from pathlib import Path

RESULTS_DIR = Path("results")
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
INCLUDE_TIER2_PRIOR = True
RERUN_VN_CANONICAL = True
RUN_CLUSTER_SPLIT = True
RUN_TIER2_MACRO = True


def _run(mod: str, *args: str) -> None:
    cmd = [sys.executable, "-u", "-m", mod, *args]
    print(">>>", " ".join(cmd))
    subprocess.run(cmd, check=True)


if Path("data/processed/vn/nlp_features.pt").exists():
    _run(
        "scripts.export_dataset_stats",
        "--data-dir", "data/processed/vn",
        "--names", "vn",
        "--k-horizon", "20",
        "--min-total-replies", "25",
        "--polar-target-mode", "future_only",
        "--edge-dim", "2",
        "--output-table", "results/tables_dataset_eda_vn.tex",
        "--output-json", "results/results_dataset_eda_vn.json",
    )

    _prior_flags = ("--include-tier2-prior",) if INCLUDE_TIER2_PRIOR else ()

    if RERUN_VN_CANONICAL:
        _run(
            "scripts.run_experiments",
            "--data-dir", "data/processed/vn",
            "--k-horizon", "20",
            "--hidden-dim", "64",
            "--edge-dim", "2",
            "--use-cascade-stats",
            *_prior_flags,
            "--min-total-replies", "25",
            "--polar-target-mode", "future_only",
            "--epochs", "100",
            "--patience", "10",
            "--output-table", "results/tables_main_vn.tex",
            "--output-json", "results/results_main_vn.json",
            "--figures-dir", "results/figures/vn",
            "--ckpt-out", "checkpoints/best_sbifagat_vn.pt",
        )

    if RUN_CLUSTER_SPLIT:
        _run(
            "scripts.run_experiments",
            "--data-dir", "data/processed/vn",
            "--k-horizon", "20",
            "--hidden-dim", "64",
            "--edge-dim", "2",
            "--use-cascade-stats",
            *_prior_flags,
            "--min-total-replies", "25",
            "--polar-target-mode", "future_only",
            "--split-mode", "embedding_cluster",
            "--edge-source", "teacher",
            "--epochs", "100",
            "--patience", "10",
            "--output-table", "results/tables_cluster_vn.tex",
            "--output-json", "results/results_cluster_vn.json",
            "--skip-figures",
        )

    _run(
        "scripts.run_experiments",
        "--data-dir", "data/processed/vn",
        "--k-horizon", "20",
        "--hidden-dim", "64",
        "--edge-dim", "2",
        "--use-cascade-stats",
        *_prior_flags,
        "--min-total-replies", "25",
        "--polar-target-mode", "future_only",
        "--split-mode", "temporal",
        "--edge-source", "teacher",
        "--epochs", "100",
        "--patience", "10",
        "--output-table", "results/tables_temporal_vn.tex",
        "--output-json", "results/results_temporal_vn.json",
        "--skip-figures",
    )

    _run(
        "scripts.run_ablation",
        "--data-dir", "data/processed/vn",
        "--k-horizon", "20",
        "--hidden-dim", "64",
        "--edge-dim", "2",
        "--use-cascade-stats",
        "--min-total-replies", "25",
        "--polar-target-mode", "future_only",
        "--epochs", "100",
        "--patience", "10",
        "--output-table", "results/tables_ablation_vn.tex",
        "--output-json", "results/results_ablation_vn.json",
        "--figures-dir", "results/figures/vn",
    )

    vn_ckpt = Path("checkpoints/best_sbifagat_vn.pt")
    if vn_ckpt.exists():
        _run(
            "scripts.analyze_mechanisms",
            "--data-dir", "data/processed/vn",
            "--ckpt", str(vn_ckpt),
            "--k-horizon", "20",
            "--min-total-replies", "25",
            "--polar-target-mode", "future_only",
            "--output-table", "results/tables_mechanisms_vn.tex",
            "--output-json", "results/results_mechanisms_vn.json",
            "--figures-dir", "results/figures/vn",
        )
        _run(
            "scripts.benchmark_efficiency",
            "--include-e2e",
            "--use-cascade-stats",
            "--k-horizon", "20",
            "--hidden-dim", "64",
            "--edge-dim", "2",
            "--ce-ckpt", "checkpoints/dual_ce_vn",
            "--gnn-ckpt", str(vn_ckpt),
            "--output-table", "results/tables_efficiency_vn.tex",
            "--output-json", "results/results_efficiency_vn.json",
        )

    if RUN_TIER2_MACRO:
        _run(
            "scripts.run_macro_signed_network",
            "--data-dir", "data/processed/vn",
            "--dataset-name", "vn",
            "--batch-size", "2048",
            "--mp-graph-mode", "coalesced",
            "--paper-dir", "results",
            "--figures-dir", "results/figures/vn",
        )
else:
    print("Skipping VN benchmarks: data/processed/vn/nlp_features.pt not found.")


In [ ]:
import subprocess
import sys
from pathlib import Path

import torch

pheme_feat_path = Path("data/processed/pheme/nlp_features.pt")
if pheme_feat_path.exists():
    _pheme_payload = torch.load(pheme_feat_path, map_location="cpu", weights_only=False)
    _meta = _pheme_payload.get("_meta", {}) or {}
    _extracted_min_replies = int(
        _pheme_payload.get("min_total_replies_extracted", _meta.get("min_total_replies_extracted", 15))
    )
    del _pheme_payload
    if _extracted_min_replies > 15:
        print(
            f"Skipping optional Track 2 PHEME Tier 2A (min_total_replies=15): "
            f"attached pheme/nlp_features.pt was extracted at min_total_replies_extracted={_extracted_min_replies} (legacy Cohort B)."
        )
    else:
        _prior_flags = ("--include-tier2-prior",) if globals().get("INCLUDE_TIER2_PRIOR", True) else ()
        subprocess.run([
            sys.executable, "-u", "-m", "scripts.export_dataset_stats",
            "--data-dir", "data/processed/pheme",
            "--names", "pheme",
            "--k-horizon", "10",
            "--min-total-replies", "15",
            "--polar-target-mode", "future_only",
            "--edge-dim", "2",
            "--output-table", "results/tables_dataset_eda_pheme.tex",
            "--output-json", "results/results_dataset_eda_pheme.json",
        ], check=True)
        subprocess.run([
            sys.executable, "-u", "-m", "scripts.run_experiments",
            "--data-dir", "data/processed/pheme",
            "--k-horizon", "10",
            "--hidden-dim", "64",
            "--edge-dim", "2",
            "--use-cascade-stats",
            *_prior_flags,
            "--min-total-replies", "15",
            "--polar-target-mode", "future_only",
            "--epochs", "100",
            "--patience", "10",
            "--output-table", "results/tables_main_pheme.tex",
            "--output-json", "results/results_main_pheme.json",
            "--figures-dir", "results/figures/pheme",
            "--ckpt-out", "checkpoints/best_sbifagat_pheme.pt",
        ], check=True)
        subprocess.run([
            sys.executable, "-u", "-m", "scripts.run_experiments",
            "--data-dir", "data/processed/pheme",
            "--k-horizon", "10",
            "--hidden-dim", "64",
            "--edge-dim", "2",
            "--use-cascade-stats",
            *_prior_flags,
            "--min-total-replies", "15",
            "--polar-target-mode", "future_only",
            "--eval-protocol", "loo",
            "--edge-source", "teacher",
            "--epochs", "100",
            "--patience", "10",
            "--output-table", "results/tables_loo_pheme.tex",
            "--output-json", "results/results_loo_pheme.json",
            "--ckpt-out", "checkpoints/best_sbifagat_pheme_loo.pt",
            "--skip-figures",
        ], check=True)
        subprocess.run([
            sys.executable, "-u", "-m", "scripts.run_ablation",
            "--data-dir", "data/processed/pheme",
            "--k-horizon", "10",
            "--hidden-dim", "64",
            "--edge-dim", "2",
            "--use-cascade-stats",
            "--min-total-replies", "15",
            "--polar-target-mode", "future_only",
            "--epochs", "100",
            "--patience", "10",
            "--output-table", "results/tables_ablation_pheme.tex",
            "--output-json", "results/results_ablation_pheme.json",
            "--figures-dir", "results/figures/pheme",
        ], check=True)
        pheme_ckpt = Path("checkpoints/best_sbifagat_pheme.pt")
        if pheme_ckpt.exists():
            subprocess.run([
                sys.executable, "-u", "-m", "scripts.analyze_mechanisms",
                "--data-dir", "data/processed/pheme",
                "--ckpt", str(pheme_ckpt),
                "--k-horizon", "10",
                "--min-total-replies", "15",
                "--polar-target-mode", "future_only",
                "--output-table", "results/tables_mechanisms_pheme.tex",
                "--output-json", "results/results_mechanisms_pheme.json",
                "--figures-dir", "results/figures/pheme",
            ], check=True)
            subprocess.run([
                sys.executable, "-u", "-m", "scripts.benchmark_efficiency",
                "--include-e2e",
                "--use-cascade-stats",
                "--k-horizon", "10",
                "--hidden-dim", "64",
                "--edge-dim", "2",
                "--ce-ckpt", "checkpoints/dual_ce_pheme",
                "--gnn-ckpt", str(pheme_ckpt),
                "--output-table", "results/tables_efficiency_pheme.tex",
                "--output-json", "results/results_efficiency_pheme.json",
            ], check=True)
        if globals().get("RUN_TIER2_MACRO", True):
            subprocess.run([
                sys.executable, "-u", "-m", "scripts.run_macro_signed_network",
                "--data-dir", "data/processed/pheme",
                "--dataset-name", "pheme",
                "--batch-size", "2048",
                "--mp-graph-mode", "coalesced",
                "--paper-dir", "results",
                "--figures-dir", "results/figures/pheme",
            ], check=True)
else:
    print("Skipping optional Track 2 PHEME Tier 2A: data/processed/pheme/nlp_features.pt not found.")


In [ ]:
import shutil
import subprocess
import sys
import zipfile
from pathlib import Path

INCLUDE_HEAVY_WEIGHTS = True

subprocess.run([
    sys.executable, "-u", "-m", "scripts.assemble_paper_artifacts",
    "--paper-dir", "results",
], check=True)

BUNDLE_NAME = Path("/kaggle/working/sbifagat_v3_gpu_results.zip") if Path("/kaggle/working").exists() else Path("sbifagat_v3_gpu_results.zip")
FORBIDDEN_SUBSTRS = ("_machine_key", ".secret", ".env")
FORBIDDEN_EXTS = {".parquet", ".zip", ".bak"}
HEAVY_MODEL_EXTS = {".safetensors", ".bin"}

added = []
with zipfile.ZipFile(BUNDLE_NAME, "w", compression=zipfile.ZIP_DEFLATED) as zf:
    for root_name in ("results", "checkpoints"):
        root_dir = Path(root_name)
        if not root_dir.exists():
            continue
        for p in sorted(root_dir.rglob("*")):
            if not p.is_file():
                continue
            low = str(p).lower()
            if any(s in low for s in FORBIDDEN_SUBSTRS) or p.suffix.lower() in FORBIDDEN_EXTS:
                continue
            if p.name == "nlp_features.pt":
                continue
            if not INCLUDE_HEAVY_WEIGHTS and p.suffix.lower() in HEAVY_MODEL_EXTS:
                continue
            zf.write(p, arcname=p.as_posix())
            added.append(p.as_posix())

print(f"Packaged {len(added)} result files into {BUNDLE_NAME.resolve()}")
content_zip = Path("/content/sbifagat_v3_gpu_results.zip")
if not Path("/kaggle/working").exists() and Path("/content").exists() and BUNDLE_NAME.resolve() != content_zip.resolve():
    shutil.copy2(BUNDLE_NAME, content_zip)
    print(f"Mirrored results bundle to {content_zip}")
if not Path("/kaggle/working").exists():
    try:
        from google.colab import files  # type: ignore
        files.download(str(BUNDLE_NAME))
    except (ImportError, NotImplementedError, RuntimeError):
        pass
